# Inteligencia de Negocios - Semana 2: Pair Programming
## Modelado Relacional y ETL Básico

**Docente:** Dudbil Olvasada Pabón Riaño
**Universidad:** Universidad Autónoma de Bucaramanga (UNAB)

---

### Objetivo de la Práctica
En esta sesión, asumiremos el rol de Ingenieros de Datos. Transformaremos un archivo CSV "sucio" de transacciones globales de ventas en un modelo relacional eficiente (**Esquema Estrella**) utilizando una base de datos SQLite en memoria y Pandas.

### 1. Simulación de Datos Empresariales (Data Warehouse Mock)
Primero, vamos a generar un DataFrame realista con miles de transacciones de una empresa tecnológica global.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Generación de datos realistas (no toy-data)
np.random.seed(42)
n_rows = 5000

data = {
    'Transaction_ID': range(1, n_rows + 1),
    'Customer_Name': np.random.choice(['TechCorp', 'Innova Inc.', 'Global Solutions', 'DataMinds', 'EduSoft', 'HealthPlus', None], n_rows),
    'Customer_Segment': np.random.choice(['Corporate', 'Consumer', 'Small Business', 'Home Office'], n_rows),
    'Product_Category': np.random.choice(['Hardware', 'Software', 'Services', 'Cloud Storage'], n_rows),
    'Region': np.random.choice(['North America', 'EMEA', 'LATAM', 'APAC'], n_rows),
    'Sales_Amount': np.round(np.random.uniform(50, 5000, n_rows), 2),
    'Discount': np.random.choice([0.0, 0.1, 0.15, 0.2, 0.25], n_rows),
    'Order_Date': pd.to_datetime(np.random.choice(pd.date_range('2024-01-01', '2026-08-01'), n_rows))
}

df_raw = pd.DataFrame(data)

# Inyectando "basura" para simular datos del mundo real (ETL necessity)
df_raw.loc[10:50, 'Sales_Amount'] = -100 # Ventas negativas anómalas
df_raw.loc[100:150, 'Region'] = 'n/a' # Strings inválidos

print("Datos crudos generados exitosamente. Vista previa:")
df_raw.head()

### 2. Proceso ETL: Limpieza de Datos (Transform)
Como analistas, no podemos construir Dashboards con ventas negativas o clientes nulos. Debemos aplicar limpieza estructurada.

In [ ]:
def clean_sales_data(df):
    # Crear una copia para no alterar la fuente original
    df_clean = df.copy()
    
    # 1. Eliminar filas donde el Cliente es Nulo (Missing Values)
    df_clean.dropna(subset=['Customer_Name'], inplace=True)
    
    # 2. Corregir ventas negativas (Outliers / Bad Data)
    df_clean['Sales_Amount'] = df_clean['Sales_Amount'].apply(lambda x: abs(x) if x < 0 else x)
    
    # 3. Reemplazar regiones 'n/a' por 'Unknown'
    df_clean['Region'] = df_clean['Region'].replace('n/a', 'Unknown')
    
    # 4. Crear columna de Venta Neta (Ingreso Real)
    df_clean['Net_Sales'] = df_clean['Sales_Amount'] * (1 - df_clean['Discount'])
    
    return df_clean

df_cleaned = clean_sales_data(df_raw)
print(f"Filas antes: {len(df_raw)} | Filas después: {len(df_cleaned)}")
df_cleaned.head()

### 3. Modelado Relacional (Esquema Estrella)
Actualmente tenemos una "Sábana de Datos" (Flat Table). En Business Intelligence, esto es ineficiente. Vamos a dividir esta sábana en tablas de **Dimensiones** (Catálogos) y una tabla de **Hechos** (Transacciones).

In [ ]:
# Creación de Dimensión: Clientes
dim_clientes = df_cleaned[['Customer_Name', 'Customer_Segment', 'Region']].drop_duplicates().reset_index(drop=True)
dim_clientes['Customer_ID'] = dim_clientes.index + 1000 # Primary Key sintética

# Creación de Dimensión: Productos
dim_productos = df_cleaned[['Product_Category']].drop_duplicates().reset_index(drop=True)
dim_productos['Product_ID'] = dim_productos.index + 200 # Primary Key sintética

# Creación de Tabla de Hechos: Mapeando las Foreign Keys
fact_ventas = df_cleaned.merge(dim_clientes, on=['Customer_Name', 'Customer_Segment', 'Region'])
fact_ventas = fact_ventas.merge(dim_productos, on='Product_Category')

# Filtrando solo las columnas necesarias para los Hechos
fact_ventas = fact_ventas[['Transaction_ID', 'Customer_ID', 'Product_ID', 'Order_Date', 'Sales_Amount', 'Discount', 'Net_Sales']]

print("Esquema Estrella construido exitosamente.")
print("Muestra Dimensión Clientes:")
display(dim_clientes.head())
print("\nMuestra Tabla de Hechos:")
display(fact_ventas.head())

### 4. Carga a Base de Datos (Load)
Simularemos la inyección de este modelo a un Data Warehouse usando SQLite.

In [ ]:
# Conectar a SQLite en memoria
conn = sqlite3.connect(':memory:')

# Guardar dataframes como tablas SQL
dim_clientes.to_sql('Dim_Clientes', conn, index=False)
dim_productos.to_sql('Dim_Productos', conn, index=False)
fact_ventas.to_sql('Fact_Ventas', conn, index=False)

print("Datos cargados en el motor SQL.")

---
## 🏆 Retos de Práctica Autónoma (Pair Programming)

Es turno de ustedes. Roten los roles de Piloto y Copiloto para resolver los siguientes retos consultando la base de datos SQL (`conn`) que acabamos de crear.

**Reto 1: Consultar el ROI por Segmento**
Escribe una consulta SQL (`pd.read_sql_query`) que una (JOIN) la tabla `Fact_Ventas` con `Dim_Clientes`. Calcula la suma total de `Net_Sales` agrupada por `Customer_Segment`. Ordena el resultado de mayor a menor.

In [ ]:
# Escribe tu solución aquí
query = """
-- SELECT ...
"""

# pd.read_sql_query(query, conn)

**Reto 2: Crear una Dimensión de Tiempo (Dim_Tiempo)**
Actualmente, la fecha en `Fact_Ventas` es cruda (`Order_Date`). Crea un nuevo DataFrame `dim_tiempo` que extraiga de `fact_ventas['Order_Date']` columnas independientes para:
- Año (`Year`)
- Mes (`Month`)
- Trimestre (`Quarter`)

Usa las funciones de `dt` de Pandas (`fact_ventas['Order_Date'].dt.year`). Luego carga este nuevo dataframe a la base de datos.

In [ ]:
# Escribe tu solución aquí

